# Sequential Architecture — LangGraph

**Pattern:** Sequential (A → B → C pipeline)  
**Framework:** LangGraph  
**Task:** Research city data → Summarize → Format travel report

## Graph Structure

```
START
  │
  ▼
┌────────────┐
│ researcher │  reads: city  →  writes: raw_data, structured_facts
└─────┬──────┘
      │
      ▼
┌────────────┐
│ summarizer │  reads: structured_facts  →  writes: summary
└─────┬──────┘
      │
      ▼
┌────────────┐
│  formatter │  reads: city, summary  →  writes: report_section
└─────┬──────┘
      │
      ▼
     END
```

**LangGraph difference from LangChain:**  
State is a shared `TypedDict` — each node reads and writes specific fields. The graph visualizes the data flow; LangChain chains just pass Python variables inline.

## Setup

In [ ]:
import os
from typing import TypedDict, Optional
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
print("✓ Ready")

## State + Mock Data

In [ ]:
class PipelineState(TypedDict):
    city: str
    raw_data: str
    structured_facts: Optional[str]
    summary: Optional[str]
    report_section: Optional[str]

def fetch_city_data(city: str) -> str:
    data = {
        "tokyo":     "Weather: Clear, 18°C. Safety: Low. Time: 22:30 JST. Highlights: Shibuya, Senso-ji.",
        "paris":     "Weather: Partly Cloudy, 16°C. Safety: Low. Time: 15:30 CET. Highlights: Eiffel Tower, Louvre.",
        "bangalore": "Weather: Rainy, 25°C. Safety: Medium. Time: 20:00 IST. Highlights: Lalbagh, Nandi Hills.",
    }
    return data.get(city.lower(), f"No data for {city}.")

print("State schema:", list(PipelineState.__annotations__.keys()))

## Three Nodes — One per Step

In [ ]:
def researcher_node(state: PipelineState) -> PipelineState:
    raw = fetch_city_data(state["city"])
    response = llm.invoke([
        SystemMessage(content="Extract and structure key travel facts."),
        HumanMessage(content=f"City: {state['city']}\nData: {raw}\n\nExtract: weather, safety, time, attractions."),
    ])
    return {"raw_data": raw, "structured_facts": response.content}

def summarizer_node(state: PipelineState) -> PipelineState:
    response = llm.invoke([
        SystemMessage(content="Write a 2-sentence traveler summary."),
        HumanMessage(content=f"Facts:\n{state['structured_facts']}"),
    ])
    return {"summary": response.content}

def formatter_node(state: PipelineState) -> PipelineState:
    response = llm.invoke([
        SystemMessage(content="Format into a polished report section with a markdown header."),
        HumanMessage(content=f"City: {state['city']}\nSummary: {state['summary']}\n\nUse '### {state['city']}' header."),
    ])
    return {"report_section": response.content}

print("3 nodes defined")

## Build and Visualize the Graph

In [ ]:
builder = StateGraph(PipelineState)
builder.add_node("researcher", researcher_node)
builder.add_node("summarizer", summarizer_node)
builder.add_node("formatter", formatter_node)

builder.add_edge(START, "researcher")
builder.add_edge("researcher", "summarizer")
builder.add_edge("summarizer", "formatter")
builder.add_edge("formatter", END)

graph = builder.compile()

# Visualize
try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

## Run the Pipeline

In [ ]:
cities = ["Tokyo", "Paris", "Bangalore"]
sections = []

for city in cities:
    print(f"\nProcessing: {city}")
    initial = PipelineState(city=city, raw_data="", structured_facts=None, summary=None, report_section=None)
    final = graph.invoke(initial)
    sections.append(final["report_section"])
    print(f"  Nodes visited: researcher → summarizer → formatter")

print("\n# Travel Report\n")
for s in sections:
    print(s)
    print()

## Stream Node-by-Node

In [ ]:
# Stream to see each node's state contribution
initial = PipelineState(city="Tokyo", raw_data="", structured_facts=None, summary=None, report_section=None)
for step in graph.stream(initial, stream_mode="updates"):
    node_name = list(step.keys())[0]
    keys_written = list(step[node_name].keys())
    print(f"  [{node_name}] wrote: {keys_written}")

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| `TypedDict` state with 5 fields | State makes data flow explicit — every node knows its contract |
| `add_edge(A, B)` enforces order | Graph structure IS the pipeline — no implicit ordering |
| `graph.stream(stream_mode="updates")` | See exactly which state fields each node writes |
| `draw_mermaid_png()` visualization | LangGraph can render the pipeline as a diagram |

### LangChain vs LangGraph for Sequential
| | LangChain | LangGraph |
|---|---|---|
| Data passing | Python variables | TypedDict state |
| Visualization | No built-in | `draw_mermaid_png()` |
| Adding branches | Refactor chain | Add conditional edge |
| Observability | Verbose logging | `stream_mode="updates"` |

**Use LangGraph when** you anticipate needing branches, loops, or checkpoints later.

### Next: [02-Parallel →](../../02-Parallel/LangGraph/parallel.ipynb)